# 04 · Refined (fato unificado, quarentena e agregados)

Aqui acontecem três coisas:

1. **Unificação** — yellow e green viram uma tabela só, `fct_taxi_trip`,
   com `trip_type` como discriminador e `pickup_datetime`/`dropoff_datetime`
   canônicos. Perguntas sobre *toda a frota* deixam de precisar de `UNION ALL`.
2. **Qualidade com quarentena** — linhas reprovadas vão para `rej_taxi_trip`
   com o motivo, em vez de sumirem num `WHERE`. Dá para auditar e reverter.
3. **Agregados** — `agg_trip_monthly` e `agg_trip_hourly`, que respondem as
   perguntas do case com uma leitura de poucas linhas.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os
import sys

_root = os.getcwd()
while _root != "/" and not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

In [0]:
from src import config
from src.processing.trusted_to_refined import TrustedToRefinedProcessor

processor = TrustedToRefinedProcessor(spark)
fato, quarentena, relatorio = processor.process_fact()

## Relatório de qualidade

Regras não bloqueantes aparecem no relatório, mas não retiram a linha do fato.

In [0]:
display(relatorio.orderBy('blocking', 'rows_failed', ascending=[False, False]))

In [0]:
%sql
-- Quantas linhas foram para a quarentena e por qual motivo.
SELECT motivo, COUNT(*) AS linhas
FROM ifood_case.refined.rej_taxi_trip
LATERAL VIEW explode(_rejection_reasons) AS motivo
GROUP BY motivo
ORDER BY linhas DESC;

## Agregados

In [0]:
monthly, hourly = processor.process_aggregates()
display(monthly)

In [0]:
display(hourly)

## Otimização física das tabelas de consumo

In [0]:
%sql
OPTIMIZE ifood_case.refined.fct_taxi_trip ZORDER BY (pickup_datetime, trip_type);
ANALYZE TABLE ifood_case.refined.fct_taxi_trip COMPUTE STATISTICS FOR ALL COLUMNS;